# **Feature Engineering — Spaceship Titanic**

**Objetivo:** Ejecutar las transformaciones y construcción de features definidas en los notebooks anteriores.

**Fundamentos — decisiones tomadas en NB02:**

| Feature | Acción | Justificación estadística (NB02) |
|---|---|---|
| `Cabin` | Extraer `Deck` / `CabinNumber` / `Side` | Deck: chi²=392.3, Side: chi²=91.1 (p<0.001) |
| `PassengerId` | Extraer `GroupSize` | chi²=145.3 (p<0.001), patrón no lineal |
| Gastos individuales | Crear `TotalSpending_Log` | r=-0.469 vs r=-0.200 original |
| `Age` | Categorizar en rangos etarios | Mann-Whitney p<0.001 |
| `VIP` | **Descartar** | corr=-0.037, solo 199 positivos de 8,490 |
| `CryoSleep` | Conservar (mayor diferenciador) | chi²=1,859.6 — señal más fuerte del dataset |

**Cadena de notebooks:**  
[NB01](01.Initial_exploration.ipynb) → [NB02](02.Analisis_Target.ipynb) → **NB03 (este)** → [NB04](04.Model_Training.ipynb)

**Input:** `data/raw/train.csv`  
**Output:** `data/processed/train_features_scaled.csv`


## **1. Importar Librerías**

In [1]:
import sys
sys.path.insert(0, '../../')  # Agrega la raíz del proyecto al path

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
import warnings

from src.features.engineering import (
    extract_cabin_features,
    extract_group_features,
    create_spending_features,
    create_age_features,
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


## **2. Cargar Datos**

In [2]:
data_path = '../../data/raw/train.csv'
df = pd.read_csv(data_path)

print(f"✅ Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head()

✅ Dataset cargado: 8,693 filas x 14 columnas


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


## **3. Verificación de Datos Faltantes**

> **NB01** identificó 12 columnas con ~2% de nulos cada una (patrón uniforme, no sistemático).
> Verificamos el estado antes de proceder y aplicamos la estrategia acordada:
>
> - Variables categóricas (`HomePlanet`, `CryoSleep`, `Destination`, `VIP`) → `'Unknown'`
> - Variables de gasto (`RoomService`, etc.) → `0` (nulo = sin gasto registrado)
> - `Age` → eliminar filas (179 registros, 2.06%) para evitar sesgo en categorización


In [3]:
missing_summary = pd.DataFrame({
    'Columna': df.columns,
    'Valores_Faltantes': df.isnull().sum(),
    'Porcentaje': (df.isnull().sum() / len(df) * 100).round(2),
    'Tipo': df.dtypes
}).sort_values('Valores_Faltantes', ascending=False)

missing_summary = missing_summary[missing_summary['Valores_Faltantes'] > 0]
print("📊 Resumen de valores faltantes:\n")
print(missing_summary.to_string(index=False))

📊 Resumen de valores faltantes:

     Columna  Valores_Faltantes  Porcentaje    Tipo
   CryoSleep                217        2.50  object
ShoppingMall                208        2.39 float64
         VIP                203        2.34  object
  HomePlanet                201        2.31  object
        Name                200        2.30  object
       Cabin                199        2.29  object
      VRDeck                188        2.16 float64
   FoodCourt                183        2.11 float64
         Spa                183        2.11 float64
 Destination                182        2.09  object
 RoomService                181        2.08 float64
         Age                179        2.06 float64


## **4. Ingeniería de Features**

Las transformaciones de esta sección ejecutan las decisiones de NB02.  
La lógica de cada función vive en `src/features/engineering.py`.

### **4.1 Extraer Features de Cabin**


In [4]:
df_clean = df.copy()

print("✅ Copia del dataset creada para procesamiento")

✅ Copia del dataset creada para procesamiento


In [5]:
# NB02: Deck (chi²=392.3) y Side (chi²=91.1) → discriminadores significativos
df_clean = extract_cabin_features(df_clean)

print('Features extraídas de Cabin: Deck, CabinNumber, Side')
print('Distribución de Deck:')
print(df_clean['Deck'].value_counts())


Features extraídas de Cabin: Deck, CabinNumber, Side
Distribución de Deck:
Deck
F          2794
G          2559
E           876
B           779
C           747
D           478
A           256
Unknown     199
T             5
Name: count, dtype: int64


In [6]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Distribución por Deck', 'Distribución por Side'))

deck_counts = df_clean['Deck'].value_counts()
fig.add_trace(go.Bar(x=deck_counts.index, y=deck_counts.values, marker_color='#2ecc71'), row=1, col=1)

side_counts = df_clean['Side'].value_counts()
fig.add_trace(go.Bar(x=side_counts.index, y=side_counts.values, marker_color='#e74c3c'), row=1, col=2)

fig.update_layout(height=400, showlegend=False)
fig.update_xaxes(title_text="Deck", row=1, col=1)
fig.update_xaxes(title_text="Side", row=1, col=2)
fig.update_yaxes(title_text="Frecuencia", row=1, col=1)
fig.show()

### **4.2 Extraer Features de Grupo de Viaje**

In [7]:
# NB02: GroupSize (chi²=145.3) — grupos de 3-6 con mayor tasa de transporte
df_clean = extract_group_features(df_clean)

print('Features de grupo creadas: TravelGroup, GroupSize')
print('Distribución de GroupSize:')
print(df_clean['GroupSize'].value_counts().sort_index())


Features de grupo creadas: TravelGroup, GroupSize
Distribución de GroupSize:
GroupSize
1    4805
2    1682
3    1020
4     412
5     265
6     174
7     231
8     104
Name: count, dtype: int64


### **4.3 Crear Feature de Gasto Total**

In [8]:
# NB02: TotalSpending_Log es el feature más correlacionado: r=-0.469 vs r=-0.200 original
df_clean = create_spending_features(df_clean)

print('Features de gasto: TotalSpending, HasSpending, SpendingCategories, TotalSpending_Log')
print(df_clean[['TotalSpending', 'TotalSpending_Log']].describe())


Features de gasto: TotalSpending, HasSpending, SpendingCategories, TotalSpending_Log
       TotalSpending  TotalSpending_Log
count    8693.000000        8693.000000
mean     1440.866329           4.253005
std      2803.045694           3.689350
min         0.000000           0.000000
25%         0.000000           0.000000
50%       716.000000           6.575076
75%      1441.000000           7.273786
max     35987.000000          10.490941


In [9]:
fig = px.histogram(df_clean, x='TotalSpending', nbins=50,
                   title='Distribución del Gasto Total',
                   labels={'TotalSpending': 'Gasto Total', 'count': 'Frecuencia'},
                   color_discrete_sequence=['#9b59b6'])
fig.update_layout(height=400)
fig.show()

### **4.4 Crear Categorías de Edad**

In [10]:
# NB02: Age significativa (Mann-Whitney p<0.001); se categoriza para capturar no-linealidad
# Rangos: Child(<13), Teen(13-18), YoungAdult(18-30), Adult(30-60), Senior(60+)
df_clean = create_age_features(df_clean)

print('Categorías de edad:')
print(df_clean['AgeCategory'].value_counts())


Categorías de edad:
AgeCategory
YoungAdult    3375
Adult         3340
Child          806
Teen           739
Senior         254
Unknown        179
Name: count, dtype: int64


In [11]:
age_cat_counts = df_clean['AgeCategory'].value_counts()
fig = px.bar(x=age_cat_counts.index, y=age_cat_counts.values,
             title='Distribución de Categorías de Edad',
             labels={'x': 'Categoría de Edad', 'y': 'Frecuencia'},
             color_discrete_sequence=['#f39c12'])
fig.update_layout(height=400)
fig.show()

## **5. Tratamiento de Valores Nulos**

### **5.1 Variables Categóricas → Unknown**

In [12]:
# Estrategia de nulos — NB01: distribución uniforme ~2%; NB02: nulos en CryoSleep/VIP son informativos
# Categóricas → 'Unknown' (preservar como categoría en el modelo)
df_clean['HomePlanet'].fillna('Unknown', inplace=True)
df_clean['CryoSleep'].fillna('Unknown', inplace=True)
df_clean['Destination'].fillna('Unknown', inplace=True)

# Variables de gasto → 0 (nulo = no utilizó el servicio)
spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in spending_cols:
    df_clean[col].fillna(0, inplace=True)

print('Imputaciones aplicadas:')
for col in ['HomePlanet', 'CryoSleep', 'Destination']:
    n = df_clean[col].value_counts().get('Unknown', 0)
    print(f'  {col}: {n} registros con Unknown')
print(f'  Spending ({len(spending_cols)} cols): nulos → 0')


Imputaciones aplicadas:
  HomePlanet: 201 registros con Unknown
  CryoSleep: 217 registros con Unknown
  Destination: 182 registros con Unknown
  Spending (5 cols): nulos → 0


### **5.2 Variables Numéricas → Solo Age se elimina**

In [13]:
rows_before = len(df_clean)
print(f"Nulos en Age: {df_clean['Age'].isnull().sum()}")

df_clean.dropna(subset=["Age"], inplace=True)

# Recomputar GroupSize con las filas que quedaron
# (algunos miembros del grupo pueden haber sido eliminados)
df_clean['GroupSize'] = df_clean.groupby('TravelGroup')['TravelGroup'].transform('count')

rows_after = len(df_clean)
print(f"""
Registros eliminados : {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.1f}%)
Registros restantes  : {rows_after:,} ({rows_after/rows_before*100:.1f}%)
""")

Nulos en Age: 179

Registros eliminados : 179 (2.1%)
Registros restantes  : 8,514 (97.9%)



### **5.3 Verificar Registros Restantes**

In [14]:
remaining_missing = df_clean.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if len(remaining_missing) > 0:
    print("⚠️ Valores faltantes restantes:")
    print(remaining_missing)
else:
    print("✅ No quedan valores faltantes en el dataset")

⚠️ Valores faltantes restantes:
Cabin    195
VIP      197
Name     197
dtype: int64


## **6. Tratamiento de Outliers**

### **6.1 Detección de Outliers en Variables Numéricas**

In [15]:
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

numeric_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpending']

print("📊 Detección de outliers (método IQR):\n")
outlier_summary = []

for col in numeric_cols:
    n_outliers, lower, upper = detect_outliers_iqr(df_clean, col)
    pct_outliers = (n_outliers / len(df_clean)) * 100
    outlier_summary.append({
        'Variable': col,
        'N_Outliers': n_outliers,
        'Porcentaje': f"{pct_outliers:.2f}%",
        'Lower_Bound': f"{lower:.2f}",
        'Upper_Bound': f"{upper:.2f}"
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))

📊 Detección de outliers (método IQR):

     Variable  N_Outliers Porcentaje Lower_Bound Upper_Bound
          Age          77      0.90%       -9.50       66.50
  RoomService        1860     21.85%      -63.00      105.00
    FoodCourt        1878     22.06%      -93.00      155.00
 ShoppingMall        1846     21.68%      -34.50       57.50
          Spa        1791     21.04%      -81.00      135.00
       VRDeck        1817     21.34%      -60.00      100.00
TotalSpending         915     10.75%    -2167.12     3611.88


### **6.2 Visualización de Outliers**

In [16]:
for col in numeric_cols:
    fig = px.box(df_clean, y=col,
                 title=f'Boxplot de {col} - Detección de Outliers',
                 labels={col: col},
                 color_discrete_sequence=['#e74c3c'])
    fig.update_layout(height=400)
    fig.show()

### **6.3 Estrategia de Tratamiento de Outliers**

In [17]:
# NB02: outliers en gasto son señal real — Mann-Whitney p<0.001 para todas las variables de gasto
# No se eliminan; la transformación log1p (ya aplicada en sección 4.3) reduce la asimetría
print('Estrategia de outliers:')
print('  Variables de gasto: MANTENER (valores extremos discriminan, confirmado en NB02)')
print('  Age: MANTENER (edades extremas son válidas)')
print('  Transformación log1p ya aplicada a TotalSpending en sección 4.3')
print()
print('  Mejora de correlación con target (NB02):')
print('    TotalSpending     → r = -0.200')
print('    TotalSpending_Log → r = -0.469')
print(f'\n  Distribución post-transformación:')
print(f"    TotalSpending     → media: {df_clean['TotalSpending'].mean():.2f}, std: {df_clean['TotalSpending'].std():.2f}")
print(f"    TotalSpending_Log → media: {df_clean['TotalSpending_Log'].mean():.2f}, std: {df_clean['TotalSpending_Log'].std():.2f}")


Estrategia de outliers:
  Variables de gasto: MANTENER (valores extremos discriminan, confirmado en NB02)
  Age: MANTENER (edades extremas son válidas)
  Transformación log1p ya aplicada a TotalSpending en sección 4.3

  Mejora de correlación con target (NB02):
    TotalSpending     → r = -0.200
    TotalSpending_Log → r = -0.469

  Distribución post-transformación:
    TotalSpending     → media: 1443.89, std: 2801.89
    TotalSpending_Log → media: 4.26, std: 3.69


In [18]:
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=('TotalSpending Original', 'TotalSpending Log'))

fig.add_trace(go.Histogram(x=df_clean['TotalSpending'], nbinsx=50, 
                           marker_color='#3498db', name='Original'), row=1, col=1)
fig.add_trace(go.Histogram(x=df_clean['TotalSpending_Log'], nbinsx=50, 
                           marker_color='#2ecc71', name='Log'), row=1, col=2)

fig.update_layout(height=400, showlegend=False)
fig.update_xaxes(title_text="TotalSpending", row=1, col=1)
fig.update_xaxes(title_text="TotalSpending_Log", row=1, col=2)
fig.update_yaxes(title_text="Frecuencia", row=1, col=1)
fig.show()

## **7. Encoding de Variables Categóricas**

### **7.1 Variables Binarias (Label Encoding)**

In [19]:
binary_cols = ['CryoSleep']

for col in binary_cols:
    df_clean[f'{col}_Encoded'] = df_clean[col].map(
        {'True': 1, 'False': 0, True: 1, False: 0, 'Unknown': -1}
    )
    print(f"{col} → {col}_Encoded  (True=1, False=0, Unknown=-1)")

df_clean['Side_Encoded'] = df_clean['Side'].map({'P': 0, 'S': 1, 'Unknown': -1})
print("Side → Side_Encoded  (P=0, S=1, Unknown=-1)")

CryoSleep → CryoSleep_Encoded  (True=1, False=0, Unknown=-1)
Side → Side_Encoded  (P=0, S=1, Unknown=-1)


### **7.2 Variables Categóricas (One-Hot Encoding)**

In [20]:
categorical_cols = ['HomePlanet', 'Destination', 'Deck', 'AgeCategory']

df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, prefix=categorical_cols, drop_first=False)

print("✅ One-Hot Encoding aplicado a:")
for col in categorical_cols:
    encoded_cols = [c for c in df_encoded.columns if c.startswith(f"{col}_")]
    print(f"   - {col}: {len(encoded_cols)} columnas creadas")

print(f"\n📊 Dimensiones después de encoding: {df_encoded.shape[0]:,} filas x {df_encoded.shape[1]} columnas")

✅ One-Hot Encoding aplicado a:
   - HomePlanet: 4 columnas creadas
   - Destination: 4 columnas creadas
   - Deck: 9 columnas creadas
   - AgeCategory: 5 columnas creadas

📊 Dimensiones después de encoding: 8,514 filas x 44 columnas


## **8. Selección de Features para Modelado**

In [21]:
# Feature selection basada en el análisis estadístico de NB02
features_to_drop = [
    'PassengerId', 'Name', 'Cabin', 'TravelGroup',  # Originales ya procesadas/extraídas
    'CryoSleep',        # Reemplazada por CryoSleep_Encoded
    'VIP',              # NB02: corr=-0.037, solo 199 positivos → descartada
    'Side',             # NB02: Deck (chi²=392) > Side (chi²=91); se reduce dimensionalidad
    'TotalSpending',    # Reemplazada por TotalSpending_Log (r: -0.200 → -0.469)
]

df_final = df_encoded.drop(columns=[col for col in features_to_drop if col in df_encoded.columns])

target = 'Transported'
X = df_final.drop(columns=[target])
y = df_final[target]

print('Features finales para modelado:')
print(f'  Total features: {X.shape[1]}')
print(f'  Muestras: {X.shape[0]:,}')
print(f'\nPrimeras 10 features: {X.columns.tolist()[:10]}')


Features finales para modelado:
  Total features: 35
  Muestras: 8,514

Primeras 10 features: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'CabinNumber', 'GroupSize', 'HasSpending', 'SpendingCategories']


## **9. Escalado de Features Numéricas**

In [22]:
numeric_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 
                    'GroupSize', 'CabinNumber', 'TotalSpending_Log', 'SpendingCategories']

numeric_features_present = [col for col in numeric_features if col in X.columns]

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_features_present] = scaler.fit_transform(X[numeric_features_present])

print("✅ Escalado aplicado con StandardScaler")
print(f"   - Features escaladas: {len(numeric_features_present)}")
print(f"\n📊 Estadísticas después del escalado:")
print(X_scaled[numeric_features_present].describe())

✅ Escalado aplicado con StandardScaler
   - Features escaladas: 10

📊 Estadísticas después del escalado:
                Age   RoomService     FoodCourt  ShoppingMall           Spa  \
count  8.514000e+03  8.514000e+03  8.514000e+03  8.514000e+03  8.514000e+03   
mean   5.800179e-17  5.226420e-17 -1.397885e-17 -2.378490e-17 -3.546872e-17   
std    1.000059e+00  1.000059e+00  1.000059e+00  1.000059e+00  1.000059e+00   
min   -1.989756e+00 -3.331042e-01 -2.811074e-01 -2.839648e-01 -2.709430e-01   
25%   -6.783417e-01 -3.331042e-01 -2.811074e-01 -2.839648e-01 -2.709430e-01   
50%   -1.261671e-01 -3.331042e-01 -2.811074e-01 -2.839648e-01 -2.709430e-01   
75%    6.330730e-01 -2.698417e-01 -2.423166e-01 -2.457805e-01 -2.227574e-01   
max    3.462968e+00  2.124694e+01  1.837167e+01  3.871716e+01  1.972429e+01   

             VRDeck     GroupSize   CabinNumber  TotalSpending_Log  \
count  8.514000e+03  8.514000e+03  8.514000e+03       8.514000e+03   
mean   1.241405e-17  1.685807e-16  2.670586

## **10. Guardar Dataset Procesado**

In [23]:
output_path_clean = '../../data/processed/train_clean.csv'
output_path_features = '../../data/processed/train_features.csv'
output_path_scaled = '../../data/processed/train_features_scaled.csv'

df_clean.to_csv(output_path_clean, index=False)
print(f"✅ Dataset limpio guardado en: {output_path_clean}")

df_final.to_csv(output_path_features, index=False)
print(f"✅ Dataset con features guardado en: {output_path_features}")

X_scaled_full = X_scaled.copy()
X_scaled_full[target] = y
X_scaled_full.to_csv(output_path_scaled, index=False)
print(f"✅ Dataset escalado guardado en: {output_path_scaled}")

✅ Dataset limpio guardado en: ../../data/processed/train_clean.csv
✅ Dataset con features guardado en: ../../data/processed/train_features.csv
✅ Dataset escalado guardado en: ../../data/processed/train_scaled.csv


## **11. Resumen de Feature Engineering**

In [24]:
print("="*70)
print("RESUMEN DE FEATURE ENGINEERING")
print("="*70)
print(f" FEATURES CREADAS:")
print(f"   - Deck, CabinNumber, Side (de Cabin)")
print(f"   - TravelGroup, GroupSize (de PassengerId)")
print(f"   - TotalSpending, HasSpending, SpendingCategories")
print(f"   - AgeCategory (categorizacion de Age)")
print(f"   - TotalSpending_Log (transformacion logaritmica)")
print(f" TRATAMIENTO DE VALORES FALTANTES:")
print(f"   - Categoricas → Unknown | Spending → 0 | Age → eliminar fila")
print(f"   - Registros antes: 8,693 | Restantes: ~8,514 (97.9%) — solo se pierden nulos de Age")
print(f"   - Spending imputado con 0 (sin registro = no uso el servicio)")
print(f" VARIABLES DESCARTADAS (sin senal):")
print(f"   - VIP (corr=-0.037, solo 2.3% positivos)")
print(f"   - IsHighSpender (corr=-0.115, redundante con TotalSpending_Log)")
print(f" ENCODING:")
print(f"   - Label Encoding: CryoSleep, Side")
print(f"   - One-Hot Encoding: HomePlanet, Destination, Deck, AgeCategory")
print(f" OUTLIERS:")
print(f"   - Mantenidos (informacion valiosa)")
print(f"   - Transformacion logaritmica aplicada a TotalSpending")
print(f" ESCALADO:")
print(f"   - StandardScaler aplicado a {len(numeric_features_present)} features numericas")
print(f" DATASET FINAL:")
print(f"   - Dimensiones: {X_scaled.shape[0]:,} filas x {X_scaled.shape[1]} features")
print(f"   - Sin valores faltantes")
print(f"   - Listo para modelado")
print("" + "="*70)
print(" Proximos pasos:")
print("   1. Division train/validation/test")
print("   2. Entrenamiento de modelos baseline")
print("   3. Feature selection (si es necesario)")
print("   4. Optimizacion de hiperparametros")
print("   5. Evaluacion y seleccion del mejor modelo")
print("="*70)

RESUMEN DE FEATURE ENGINEERING
 FEATURES CREADAS:
   - Deck, CabinNumber, Side (de Cabin)
   - TravelGroup, GroupSize (de PassengerId)
   - TotalSpending, HasSpending, SpendingCategories
   - AgeCategory (categorizacion de Age)
   - TotalSpending_Log (transformacion logaritmica)
 TRATAMIENTO DE VALORES FALTANTES:
   - Categoricas → Unknown | Spending → 0 | Age → eliminar fila
   - Registros antes: 8,693 | Restantes: ~8,514 (97.9%) — solo se pierden nulos de Age
   - Spending imputado con 0 (sin registro = no uso el servicio)
 VARIABLES DESCARTADAS (sin senal):
   - VIP (corr=-0.037, solo 2.3% positivos)
   - IsHighSpender (corr=-0.115, redundante con TotalSpending_Log)
 ENCODING:
   - Label Encoding: CryoSleep, Side
   - One-Hot Encoding: HomePlanet, Destination, Deck, AgeCategory
 OUTLIERS:
   - Mantenidos (informacion valiosa)
   - Transformacion logaritmica aplicada a TotalSpending
 ESCALADO:
   - StandardScaler aplicado a 10 features numericas
 DATASET FINAL:
   - Dimensiones: 8,51